In [1]:
import sys
sys.path.append("..")  # make DS/ importable so `scratch` is found

In [2]:
# CORRELATION
# - is the relationship between two variables

# In a social app, relationship between a user's number of friends and the amount of time they spend on the app is an example of correlation.

# hypothesize the constants α and β
# such that yi = α + βxi + εi
# where;
# yi = dependent variable (time spent on the app by user i)
# xi = independent variable (number of friends of user i)
# εi = error term

In [3]:
# Assuming we know the values of α and β, we can predict the time spent on the app by a user based on their number of friends.

def predict(alpha: float, beta: float, x_i: float) -> float:
    """
    Predict the dependent variable (time spent on the app) based on the independent variable (number of friends).
    
    Parameters:
    alpha (float): The intercept of the regression line.
    beta (float): The slope of the regression line.
    x_i (float): The number of friends of the user.
    
    Returns:
    float: The predicted time spent on the app by the user.
    """
    return alpha + beta * x_i

In [4]:
# Any choice of α and β gives us a predicted output for each input x_i
# Knowing the actual output we can compute the error

def error(alpha: float, beta: float, x_i: float, y_i: float) -> float:
    """
    The error for predicting beta * y_i + alpha
    when the actual value is y_i
    """
    return predict(alpha, beta, x_i) - y_i

In [5]:
# Least Squares Solution
# - Goal is to choose the α and β that make the square of errors as small as possible
# Square errors summation for entire set

from scratch.linear_algebra import Vector

def sum_of_sqerrors(alpha: float, beta: float, x: Vector, y: Vector) -> float:
    return sum(
        error(alpha, beta, x_i, y_i) ** 2
        for x_i, y_i in zip(x, y)
    )

# To find the error minimizing α and β using calculus;

from typing import Tuple
from scratch.linear_algebra import Vector
from scratch.statistics import correlation, standard_deviation, mean

def least_squares_fit(x: Vector, y: Vector) -> Tuple[float, float]:
    """
    Given two vectors x and y,
    find the least-squares values of α and β
    """
    beta = correlation(x, y) * standard_deviation(y) / standard_deviation(x)
    alpha = mean(y) - beta * mean(x)
    return alpha, beta

# test
x = [i for i in range(-100, 110, 10)]
y = [3 * i - 5 for i in x]

# should find that y = 3x - 5
assert least_squares_fit(x, y) == (-5, 3)

## 
from scratch.statistics import num_friends_good, daily_minutes_good

alpha, beta = least_squares_fit(num_friends_good, daily_minutes_good)
assert 22.9 < alpha < 23.0
assert 0.9 < beta < 0.905

print(alpha, beta)
# 22.947552413468976 0.9038659456058725
# Our model says we expect a user with n friends to spend 22.95 + n * 0.903 minutes on the site each day
# a user with no friend still spends about 23 min a day on the site

22.947552413468976 0.9038659456058725


In [6]:
# coefficient of determination (or R-squared) 

from scratch.statistics import de_mean

def total_sum_of_squares(y: Vector) -> float:
    """the total squared variation of y_i's from their mean"""
    return sum(v ** 2 for v in de_mean(y))

def r_squared(alpha: float, beta: float, x: Vector, y: Vector) -> float:
    """
    the fraction of variation in y captured by the model, which equals
    1 - the fraction of variation in y not captured by the model
    """
    return 1.0 - (sum_of_sqerrors(alpha, beta, x, y) / total_sum_of_squares(y))

In [ ]:
# Using Gradient Descent

import random
import tqdm
from scratch.gradient_descent import gradient_step

num_epochs = 10000 # number of training iterations
random.seed(0)

guess = [random.random(), random.random()] # choose random value to start

learning_rate = 0.00001

with tqdm.trange(num_epochs) as t:
    for _ in t:
        alpha, beta = guess

        # Partial derivative of loss with respect to alpha
        grad_a = sum(2 * error(alpha, beta, x_i, y_i)
                    for x_i, y_i in zip(num_friends_good, daily_minutes_good))
        
        # Partial derivative of loss with respect to beta
        grad_b = sum(2 * error(alpha, beta, x_i, y_i) * x_i
                    for x_i, y_i in zip(num_friends_good, daily_minutes_good))

        # Compute loss to stick in the tqdm description
        loss = sum_of_sqerrors(alpha, beta, num_friends_good, daily_minutes_good)
        t.set_description(f"loss: {loss:.3f}")

        # Finally, update the guess
        guess = gradient_step(guess, [grad_a, grad_b], -learning_rate)

# We should get pretty much the same results:
alpha, beta = guess
assert 22.9 < alpha < 23.0
assert 0.9 < beta < 0.905

print(alpha, beta)

loss: 13196.619: 100%|██████████| 10000/10000 [00:08<00:00, 1237.45it/s]

22.947552155340915 0.9038659662765034
